# Pose estimation - Application

Now that we have a technique to extract information from the videos, we can apply it to all the videos in our dataset. 

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.signal import savgol_filter, find_peaks

In [2]:
data_folder = "data/sit2stand/videos_sit2stand_valid/"
list_of_videos = [f for f in os.listdir(data_folder) if f.endswith(".mp4")]

Each video is accompanied by some demographic data that tell us:
- Video Frame Rate
- Age
- Sex
- BMI
- osteoarthritis (yes or no)

all this information can be useful for our classification.

In [3]:
demographics = pd.read_csv(os.path.join("data/sit2stand/", "sit2stand_clean_dataset.csv"))
demographics.head()

,subjectid,orientation,framerate,Age,Sex,BMI,Height,Weight,OA_check,Global03,Global04,WOMAC_total
0,en9ir5DJ,L,30.000000,56,0,24.405142,73,185.0,1,3,3,25.0
1,v8Q5mj77,R,29.970030,80,0,22.148708,69,150.0,0,4,5,0.0
2,6MFubdNK,R,29.833333,51,1,23.170166,64,135.0,0,4,4,0.0
3,mgKwPHtm,R,29.970030,63,0,26.622856,67,170.0,0,3,3,0.0
4,Ni69ryle,L,30.000000,57,1,21.630769,65,130.0,0,5,5,0.0


We will process each video at the time

In [ ]:
model = YOLO("models/yolo26s-pose.pt")  # COCO 17-keypoint pose model

poses_dict = {} # i will store the results in a dictionary with the video name as key and the pose sequence as value
features_dict = {} # i will store the features in a dictionary with the video name as key
for video in list_of_videos:
    print(f"Processing video: {video}")

    #----

    #get demographic info for the video
    subjectData = demographics[demographics["subjectid"] == video[:-4]] # remove .mp4 from the video name to match with subjectid in demographics
    age = subjectData["Age"].values[0]
    gender = subjectData["Sex"].values[0]
    BMI = subjectData["BMI"].values[0]
    fps = subjectData["framerate"].values[0]
    OA_status = subjectData["OA_check"].values[0]
    Global03 = subjectData["Global03"].values[0]

    #----

    #process the video with the model to get the pose sequence

    video_path = os.path.join(data_folder, video)
    results = model.track(
                            source=video_path,
                            stream=True,
                            tracker="bytetrack.yaml",
                            persist=True,
                            verbose=False,
                            device = 'mps' if torch.backends.mps.is_available() else 'cpu'
                        )
    
    #----

    #extracting keypoints from the results

    frame_ids = []
    pose_seq = []  # (num_frames_detected, 17, 2)
    boxes_height = []
    results_list = list(results)

    #Store results in dictionary
    poses_dict[video] = {"results": results_list}

    #put get measurements out of the results
    for i, r in enumerate(results_list):
        if r.keypoints is None or len(r.keypoints.xy) == 0:
            continue

        # Pick the highest-confidence person in frame
        kpts = r.keypoints.xy.cpu().numpy()  # (n_people, 17, 2)
        conf = r.boxes.conf.cpu().numpy() if r.boxes is not None else np.ones(len(kpts))
        box_height = r.boxes.xywh.cpu().numpy()[:, 3][0] if r.boxes is not None else 0
    
        person_idx = int(np.argmax(conf))

        frame_ids.append(i)
        pose_seq.append(kpts[person_idx])
        boxes_height.append(box_height)

    #----

    #compute the vertical movement and velocity from the pose sequence

    torso_COM = np.mean(np.array(pose_seq)[:,[5,6,11,12], :], axis=1)
    ankles_center = np.mean(np.array(pose_seq)[:,[15,16], :], axis=1)  # (n_people, 2)
    vertical_movement_frames = np.abs(torso_COM[:, 1] - ankles_center[:, 1])  # Use distance between shoulders and ankles as a proxy for vertical movement

    y_smooth = savgol_filter(vertical_movement_frames, window_length=11, polyorder=3, mode='nearest', deriv=0)  # Smooth the signal to reduce noise
    y_smooth_normalized = y_smooth/np.max(boxes_height) if np.max(boxes_height) != 0 else y_smooth
    velocity_normalized = savgol_filter(y_smooth_normalized, window_length=11, polyorder=3, mode='nearest', deriv=1)*fps  # Compute velocity (first derivative)

    #---- 

    #detect peaks in the signal and extract features

    # 1) Peaks in distance (standing positions)
    dist_peaks, _ = find_peaks(
        y_smooth_normalized,
        distance=int(fps),
        prominence=0.25 * (np.max(y_smooth_normalized) - np.min(y_smooth_normalized))
    )

    # 2) Peaks in velocity (max ascent / max descent)
    vel_max_peaks, _ = find_peaks(
        velocity_normalized,
        distance=int(fps)*0.5,
        prominence=0.25 * (np.max(velocity_normalized) - np.min(velocity_normalized))
    )
    vel_min_peaks, _ = find_peaks(
        -velocity_normalized,
        distance=int(fps)*0.5,
        prominence=0.25 * (np.max(velocity_normalized) - np.min(velocity_normalized))
    )

    # --- Time base ---
    #difference between last sitting velocity peak and first standing distance peak
    total_time_sec = (vel_min_peaks[-1] - dist_peaks[0]) / fps if len(vel_min_peaks) > 0 and len(dist_peaks) > 0 else 0

    # --- Cadence and peak-spacing features ---
    time_between_dist_peaks = np.diff(dist_peaks) / fps if len(dist_peaks) > 1 else np.array([])

    cadence_per_min = (len(dist_peaks) / total_time_sec) * 60 if total_time_sec > 0 else np.nan
    avg_time_between_dist_peaks = np.mean(time_between_dist_peaks) if len(time_between_dist_peaks) else np.nan
    std_time_between_dist_peaks = np.std(time_between_dist_peaks) if len(time_between_dist_peaks) else np.nan

    # --- Velocity amplitude features ---
    standing_vel_peaks = velocity_normalized[vel_max_peaks] if len(vel_max_peaks) else np.array([])
    sitting_vel_peaks = np.abs(velocity_normalized[vel_min_peaks]) if len(vel_min_peaks) else np.array([])

    avg_max_standing_velocity = np.mean(standing_vel_peaks) if len(standing_vel_peaks) else np.nan
    std_max_standing_velocity = np.std(standing_vel_peaks) if len(standing_vel_peaks) else np.nan

    avg_max_sitting_velocity = np.mean(sitting_vel_peaks) if len(sitting_vel_peaks) else np.nan
    std_max_sitting_velocity = np.std(sitting_vel_peaks) if len(sitting_vel_peaks) else np.nan

    avg_velocity_ratio = avg_max_standing_velocity / avg_max_sitting_velocity if avg_max_sitting_velocity != 0 else np.nan
    std_velocity_ratio = np.std([avg_max_standing_velocity / avg_max_sitting_velocity for avg_max_sitting_velocity in sitting_vel_peaks if avg_max_sitting_velocity != 0]) if len(sitting_vel_peaks) else np.nan

    # --- Time between max standing velocity and subsequent max sitting velocity ---
    stand_to_sit_times = []
    for s in vel_max_peaks:
        next_sit = vel_min_peaks[vel_min_peaks > s]
        if len(next_sit):
            stand_to_sit_times.append((next_sit[0] - s) / fps)

    stand_to_sit_times = np.array(stand_to_sit_times)
    avg_time_stand_to_sit_vel = np.mean(stand_to_sit_times) if len(stand_to_sit_times) else np.nan
    std_time_stand_to_sit_vel = np.std(stand_to_sit_times) if len(stand_to_sit_times) else np.nan

    # Final feature set
    features = {
        "id" : video[:-4],
        "OA_status": OA_status,
        "age": age,
        "gender": gender,
        "BMI": BMI, 
        "Global03": Global03,
        "cadence_per_min": cadence_per_min,
        "avg_time_between_distance_peaks_s": avg_time_between_dist_peaks,
        "std_time_between_distance_peaks_s": std_time_between_dist_peaks,
        "avg_max_standing_velocity": avg_max_standing_velocity,
        "std_max_standing_velocity": std_max_standing_velocity,
        "avg_max_sitting_velocity": avg_max_sitting_velocity,
        "std_max_sitting_velocity": std_max_sitting_velocity,
        "avg_time_between_max_stand_and_sit_velocity_s": avg_time_stand_to_sit_vel,
        "std_time_between_max_stand_and_sit_velocity_s": std_time_stand_to_sit_vel,
        "avg_velocity_ratio": avg_velocity_ratio,
        "std_velocity_ratio": std_velocity_ratio,
    }

    features_df = pd.DataFrame([features])
    #accomulate the features in a dataframe for all videos
    if 'features_df_all' not in locals():
        features_df_all = features_df
    else:
        features_df_all = pd.concat([features_df_all, features_df], ignore_index=True)  

    


Processing video: 6MFubdNK.mp4


In [8]:
features_df_all

,id,OA_status,age,gender,BMI,Global03,cadence_per_min,avg_time_between_distance_peaks_s,std_time_between_distance_peaks_s,avg_max_standing_velocity,std_max_standing_velocity,avg_max_sitting_velocity,std_max_sitting_velocity,avg_time_between_max_stand_and_sit_velocity_s,std_time_between_max_stand_and_sit_velocity_s,avg_velocity_ratio,std_velocity_ratio
0,6MFubdNK,0,51,1,23.170166,4,21.936275,2.304469,0.289682,0.386034,0.033272,0.285256,0.023602,1.052514,0.034183,1.353288,0.115806
1,6MFubdNK,0,51,1,23.170166,4,31.403509,2.304469,0.289682,0.386034,0.033272,0.285256,0.023602,1.052514,0.034183,1.353288,0.115806


In [ ]:
features_df_all.to_csv('Results.csv')